# Sistem de Analiză și Predicție a Traficului Aerian
# Notebook 1 — Procesarea și analiza datelor
## 1. Introducere

### 1.1 Prezentarea setului de date
Setul de date utilizat în cadrul acestui proiect a fost obținut prin colectarea sistematică de informații despre traficul aerian, utilizând **OpenSky Network API** (https://openskynetwork.github.io/opensky-api/python.html#examples). 

Având în vedere restricțiile privind numărul limitat de credite API oferite de OpenSky Network, procesul de colectare a fost implementat într-o manieră incrementală. Astfel, scriptul de colectare **concatenează datele noi** la fișierul `history_traffic.csv` existent, prevenind suprascrierea și asigurând acumularea unui volum istoric consistent, necesar pentru antrenarea modelelor de Machine Learning.

Datele acoperă **5 aeroporturi europene majore**:
- **LROP** — Henri Coandă, București (România)
- **EGLL** — Heathrow, Londra (Marea Britanie)
- **LFPG** — Charles de Gaulle, Paris (Franța)
- **EDDF** — Frankfurt (Germania)
- **EHAM** — Amsterdam Schiphol (Olanda)

Pentru fiecare aeroport au fost colectate atât **sosirile** cât și **plecările** pe parcursul mai multor zile consecutive, folosind endpoint-urile `/flights/arrival` și `/flights/departure` ale API-ului OpenSky.

**Coloanele setului de date:**

| Coloană | Tip | Descriere |
|---|---|---|
| `icao24` | string | Adresa unică ICAO 24-bit a transponderului |
| `callsign` | string | Indicativul zborului (ex: DLH7WK) |
| `airport` | string | Codul ICAO al aeroportului monitorizat |
| `type` | string | Tipul zborului: `arrival` sau `departure` |
| `firstSeen` | long | Timestamp Unix al primei detecții |
| `lastSeen` | long | Timestamp Unix al ultimei detecții |
| `arrival_hour` | int | Ora sosirii/plecării (0-23) |
| `day_of_week` | int | Ziua săptămânii (0=Luni, 6=Duminică) |
| `estDepartureAirport` | string | Aeroportul de plecare estimat |
| `estArrivalAirport` | string | Aeroportul de sosire estimat |
| `date` | string | Data zborului (YYYY-MM-DD) |


**Exemplu de date (format CSV):**
```csv
icao24,callsign,airport,type,firstSeen,lastSeen,arrival_hour,day_of_week,estDepartureAirport,estArrivalAirport,date
3c65cd,DLH7WK,LROP,arrival,1782586842,1782593625,20,5,EDDF,LROP,2026-06-27
471f62,WMT4GW,LROP,arrival,1782582120,1782593488,20,5,LEMD,LROP,2026-06-27
4a08e9,ROT384J,LROP,arrival,1782583944,1782593360,20,5,LFPG,LROP,2026-06-27
```


### 1.2 Obiectivele proiectului

Scopul lucrării este dezvoltarea unui sistem distribuit de procesare, analiză și predicție a traficului aerian. Obiectivele specifice sunt:

- **Ingestia și procesarea datelor Big Data:** Colectarea automată și curățarea unui volum mare de date (batch și stream) utilizând Apache Spark și Apache Kafka, cu gestionarea eficientă a resurselor API.

- **Analiza exploratorie (EDA):** Identificarea pattern-urilor operaționale prin Spark SQL, precum rutele frecvente, sezonalitatea și intervalele orare de vârf din aeroporturi.

- **Predicție și modelare Machine Learning:**
  - **Regresie:** Estimarea numărului de zboruri pe oră utilizând algoritmul Random Forest.
  - **Clustering:** Segmentarea aeroporturilor în funcție de intensitatea traficului aerian folosind algoritmul K-Means.
  - **Deep Learning:** Prognozarea evoluției traficului aerian prin intermediul rețelelor neuronale de tip LSTM.

- **Monitorizare în timp real:** Implementarea unui flux de procesare live care detectează aeronavele aflate în proximitatea aeroporturilor (Geofencing), clasifică starea lor operațională (de exemplu: *Apropiere*, *Aterizare*) și generează rapoarte dinamice privind activitatea aeroportuară.

## 2. Procesarea datelor cu Apache Spark

### 2.1 Inițializarea SparkSession

Primul pas este crearea unei sesiuni Spark. Folosim `local[*]` pentru a utiliza toate core-urile disponibile pe mașina locală.

In [15]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

print(f'Python path: {sys.executable}')
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, udf, when, trim, avg, dayofweek, count, to_date, round as spark_round,
    min as spark_min, max as spark_max
)
from pyspark.sql.types import StringType, IntegerType
import os

spark = SparkSession.builder \
    .appName('AirTrafficAnalysis') \
    .master('local[*]') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print('SparkSession initializat cu succes!')

Python path: C:\Users\Lenovo\anaconda3\envs\spark_env\python.exe
Spark version: 3.5.3
SparkSession initializat cu succes!


### 2.2 Încărcarea și inspecția datelor

Încărcăm CSV-ul colectat din OpenSky Network și inspectăm structura acestuia: schema, numărul de înregistrări și primele rânduri.

In [16]:
CSV_PATH = os.path.join('..', 'data', 'history_traffic.csv')

df_raw = spark.read.csv(CSV_PATH, header=True, inferSchema=True)

print(f'Total inregistrari brute: {df_raw.count()}')
print(f'Numar coloane: {len(df_raw.columns)}')
print()
df_raw.printSchema()

Total inregistrari brute: 389410
Numar coloane: 11

root
 |-- icao24: string (nullable = true)
 |-- callsign: string (nullable = true)
 |-- airport: string (nullable = true)
 |-- type: string (nullable = true)
 |-- firstSeen: integer (nullable = true)
 |-- lastSeen: integer (nullable = true)
 |-- arrival_hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- estDepartureAirport: string (nullable = true)
 |-- estArrivalAirport: string (nullable = true)
 |-- date: date (nullable = true)



In [18]:
# Primele 10 randuri pentru a inspecta datele brute
df_raw.show(10, truncate=False)

+------+--------+-------+-------+----------+----------+------------+-----------+-------------------+-----------------+----------+
|icao24|callsign|airport|type   |firstSeen |lastSeen  |arrival_hour|day_of_week|estDepartureAirport|estArrivalAirport|date      |
+------+--------+-------+-------+----------+----------+------------+-----------+-------------------+-----------------+----------+
|3c65cd|DLH7WK  |LROP   |arrival|1782586842|1782593625|20          |5          |EDDF               |LROP             |2026-06-27|
|471f62|WMT4GW  |LROP   |arrival|1782582120|1782593488|20          |5          |LEMD               |LROP             |2026-06-27|
|4a08e9|ROT384J |LROP   |arrival|1782583944|1782593360|20          |5          |LFPG               |LROP             |2026-06-27|
|4ca708|RYR3EM  |LROP   |arrival|1782586700|1782592638|20          |5          |EDDB               |LROP             |2026-06-27|
|48ada9|LOT639  |LROP   |arrival|1782587673|1782592501|20          |5          |EPWA      

### 2.3 Curățarea datelor (DataFrame API)

Înainte de analiză, curățăm datele:
- Eliminăm spațiile din `callsign` (observate în datele brute: `DLH7WK  `)
- Eliminăm rândurile cu `arrival_hour` sau `airport` null
- Eliminăm rândurile cu `estDepartureAirport` gol (zboruri fără origine cunoscută)
- Excluderea zborurilor în care aeroportul de plecare este identic cu cel de sosire (estDepartureAirport $\neq$ estArrivalAirport)
- Filtrăm `arrival_hour` să fie în intervalul valid [0, 23]
- Eliminăm duplicatele pe baza `icao24` + `date` + `airport` + `type`

In [19]:
df_clean = df_raw \
    .withColumn('callsign', trim(col('callsign'))) \
    .dropna(subset=['arrival_hour', 'airport', 'lastSeen', 'firstSeen']) \
    .filter(col('estDepartureAirport').isNotNull() & (col('estDepartureAirport') != '')) \
    .filter(col('estArrivalAirport').isNotNull() & (col('estArrivalAirport') != '')) \
    .filter(col('estDepartureAirport') != col('estArrivalAirport')) \
    .filter(col('arrival_hour').between(0, 23)) \
    .dropDuplicates(['icao24', 'date', 'airport', 'type'])

print(f'Inregistrari dupa curatare: {df_clean.count()}')
print(f'Inregistrari eliminate:     {df_raw.count() - df_clean.count()}')

Inregistrari dupa curatare: 223565
Inregistrari eliminate:     165845


### 2.4 Transformarea datelor (DataFrame API)

Adăugăm coloane derivate utile pentru analiză și modelare:
- `flight_duration_min` — durata zborului în minute, calculată din `lastSeen` - `firstSeen`
- `is_weekend` — indicator boolean dacă zborul e în weekend (sâmbătă=5 sau duminică=6)
- `rush_period` — perioada zilei, calculată printr-o **funcție UDF definită de utilizator**

In [20]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def classify_rush(hour):
    if hour is None: return 'Unknown'
    hour = int(hour)
    if 0 <= hour <= 5: return 'Off-Peak'
    elif 6 <= hour <= 9: return 'Morning Rush'
    elif 10 <= hour <= 15: return 'Midday'
    elif 16 <= hour <= 20: return 'Evening Rush'
    else: return 'Night'

# 1. Aplicăm filtrul general (zboruri între 30m - 24)
df = df_clean \
    .withColumn('flight_duration_min', spark_round((col('lastSeen') - col('firstSeen')) / 60, 1)) \
    .filter(
        (col('flight_duration_min').between(30, 1440)) & 
        (col('estDepartureAirport') != col('estArrivalAirport'))
    ) \
    .withColumn('day_of_week', (dayofweek(to_date(col('date'), 'yyyy-MM-dd')) + 5) % 7) \
    .withColumn('is_weekend', when(col('day_of_week') >= 5, 1).otherwise(0)) \
    .withColumn('rush_period', classify_rush(col('arrival_hour')))


print(f'Total randuri dupa transformari: {df.count()}')
print(f'Coloane noi adaugate: flight_duration_min, is_weekend, rush_period, day_of_week recalculat')
df.select('callsign', 'airport', 'type', 'arrival_hour',
          'rush_period', 'is_weekend', 'flight_duration_min', 'day_of_week').show(10)

Total randuri dupa transformari: 220708
Coloane noi adaugate: flight_duration_min, is_weekend, rush_period, day_of_week recalculat
+--------+-------+---------+------------+------------+----------+-------------------+-----------+
|callsign|airport|     type|arrival_hour| rush_period|is_weekend|flight_duration_min|day_of_week|
+--------+-------+---------+------------+------------+----------+-------------------+-----------+
|  MSR778|   EGLL|departure|          19|Evening Rush|         0|              238.6|          4|
|  MSR786|   EDDF|departure|          18|Evening Rush|         0|              204.9|          4|
|  MSR780|   EGLL|departure|           1|    Off-Peak|         0|              246.7|          1|
|  MSR802|   LFPG|departure|           0|    Off-Peak|         0|              236.0|          0|
|  MSR784|   EGLL|departure|          13|      Midday|         1|              249.7|          6|
|  MSR783|   EGLL|  arrival|           7|Morning Rush|         0|              274.2|

### 2.5 Agregări cu DataFrame API

#### 2.5.1 Numărul total de zboruri per aeroport și tip

Calculăm distribuția zborurilor per aeroport, separând sosirile de plecări, pentru a înțelege volumul de trafic al fiecărui aeroport.

In [21]:
df.groupBy('airport', 'type') \
  .agg(count('*').alias('total_zboruri')) \
  .orderBy('airport', 'type') \
  .show()

+-------+---------+-------------+
|airport|     type|total_zboruri|
+-------+---------+-------------+
|   EDDF|  arrival|        23039|
|   EDDF|departure|        22785|
|   EGLL|  arrival|        28218|
|   EGLL|departure|        28432|
|   EHAM|  arrival|        25468|
|   EHAM|departure|        25379|
|   LFPG|  arrival|        26916|
|   LFPG|departure|        27399|
|   LROP|  arrival|         6519|
|   LROP|departure|         6553|
+-------+---------+-------------+



#### 2.5.2 Trafic mediu per oră per aeroport

Agregăm datele pentru a obține numărul mediu de zboruri per oră pentru fiecare aeroport. Acesta este **tabelul principal** care va fi folosit pentru antrenarea modelelor ML.

In [22]:
from pyspark.sql.functions import lag, quarter, countDistinct, substring, sin, cos, to_date, col, stddev
from pyspark.sql import Window

w_lag = Window.partitionBy('airport', 'arrival_hour').orderBy('date')

w_7days = Window.partitionBy('airport', 'arrival_hour') \
               .orderBy('date') \
               .rowsBetween(-7, -1)

# Nr rute unice din ZIUA PRECEDENTA
w_yesterday = Window.partitionBy('airport', 'arrival_hour') \
                    .orderBy('date') \
                    .rowsBetween(-1, -1)

nr_rute_daily = df.groupBy('airport', 'arrival_hour', 'date') \
                  .agg(countDistinct('estDepartureAirport').alias('nr_rute_unice_day'))

nr_companii_daily = df.withColumn('airline_code', substring(col('callsign'), 1, 3)) \
                     .groupBy('airport', 'arrival_hour', 'date') \
                     .agg(countDistinct('airline_code').alias('nr_companii_day'))

# Adaugam la df_hourly si aplicam lag pentru a folosi doar date din trecut
df_hourly = (df
    .withColumn('day_of_week', (dayofweek(to_date(col('date'), 'yyyy-MM-dd')) + 5) % 7)
    .groupBy('airport', 'arrival_hour', 'day_of_week', 'is_weekend', 'date')
    .agg(count('*').alias('flights_count'))
    .join(nr_rute_daily, on=['airport', 'arrival_hour', 'date'], how='left')
    .join(nr_companii_daily, on=['airport', 'arrival_hour', 'date'], how='left')
    .withColumn('flights_yesterday', lag('flights_count', 1).over(w_lag))
    .withColumn('media_7zile', avg('flights_count').over(w_7days))
    .withColumn('quarter', quarter(to_date(col('date'), 'yyyy-MM-dd')))
    .withColumn('nr_rute_unice', lag('nr_rute_unice_day', 1).over(w_lag))
    .withColumn('nr_companii', lag('nr_companii_day', 1).over(w_lag))
    .withColumn('hour_sin', sin(col('arrival_hour') * 3.14159 / 12))
    .withColumn('hour_cos', cos(col('arrival_hour') * 3.14159 / 12))
    .withColumn('volatilitate_7zile', stddev('flights_count').over(w_7days))
    .fillna({'volatilitate_7zile': 0})
    .drop('nr_rute_unice_day', 'nr_companii_day')
    .orderBy('airport', 'date', 'arrival_hour')
)

print(f'df_hourly fara data leakage: {df_hourly.count()} randuri')
df_hourly.select('airport', 'arrival_hour', 'date', 'flights_count',
                 'flights_yesterday', 'nr_rute_unice', 'nr_companii').show(20)

df_hourly fara data leakage: 8088 randuri
+-------+------------+----------+-------------+-----------------+-------------+-----------+
|airport|arrival_hour|      date|flights_count|flights_yesterday|nr_rute_unice|nr_companii|
+-------+------------+----------+-------------+-----------------+-------------+-----------+
|   EDDF|           0|2026-04-26|            5|             NULL|         NULL|       NULL|
|   EDDF|           1|2026-04-26|            3|             NULL|         NULL|       NULL|
|   EDDF|           2|2026-04-26|            5|             NULL|         NULL|       NULL|
|   EDDF|           3|2026-04-26|           11|             NULL|         NULL|       NULL|
|   EDDF|           4|2026-04-26|           14|             NULL|         NULL|       NULL|
|   EDDF|           5|2026-04-26|           20|             NULL|         NULL|       NULL|
|   EDDF|           6|2026-04-26|           27|             NULL|         NULL|       NULL|
|   EDDF|           7|2026-04-26|     

#### 2.5.3 Durata medie a zborurilor per aeroport

Calculăm durata medie, minimă și maximă a zborurilor pentru fiecare aeroport, pentru a înțelege profilul rutelor deservite.

In [23]:
from pyspark.sql.functions import first

# Durata medie, min, max per aeroport
df.groupBy('airport') \
  .agg(
      spark_round(avg('flight_duration_min'), 1).alias('durata_medie_min'),
      spark_round(spark_min('flight_duration_min'), 1).alias('durata_min_min'),
      spark_round(spark_max('flight_duration_min'), 1).alias('durata_max_min')
  ) \
  .orderBy('durata_medie_min', ascending=False) \
  .show()


+-------+----------------+--------------+--------------+
|airport|durata_medie_min|durata_min_min|durata_max_min|
+-------+----------------+--------------+--------------+
|   EGLL|           301.2|          30.0|        1438.0|
|   EDDF|           227.8|          30.0|        1438.9|
|   LFPG|           224.6|          30.0|        1438.4|
|   EHAM|           197.0|          30.0|        1439.2|
|   LROP|           127.5|          30.0|        1372.0|
+-------+----------------+--------------+--------------+



#### 2.5.4 Cel mai scurt / lung zbor detectat pentru fiecare aeroport

In [25]:
# Zborul cel mai scurt per aeroport
print("=== Cel mai scurt zbor detectat per aeroport ===")
from pyspark.sql import Window
from pyspark.sql.functions import rank

w_min = Window.partitionBy('airport').orderBy('flight_duration_min')
df.withColumn('rank', rank().over(w_min)) \
  .filter(col('rank') == 1) \
  .select('airport', 'callsign', 'type', 'estDepartureAirport', 
          'estArrivalAirport', 'flight_duration_min') \
  .orderBy('airport') \
  .show()

print("=== Cel mai lung zbor detectat per aeroport ===")
w_max = Window.partitionBy('airport').orderBy(col('flight_duration_min').desc())
df.withColumn('rank', rank().over(w_max)) \
  .filter(col('rank') == 1) \
  .select('airport', 'callsign', 'type', 'estDepartureAirport',
          'estArrivalAirport', 'flight_duration_min') \
  .orderBy('airport') \
  .show()

=== Cel mai scurt zbor detectat per aeroport ===
+-------+--------+---------+-------------------+-----------------+-------------------+
|airport|callsign|     type|estDepartureAirport|estArrivalAirport|flight_duration_min|
+-------+--------+---------+-------------------+-----------------+-------------------+
|   EDDF|  LHX076|departure|               EDDF|             EDDL|               30.0|
|   EDDF|  DLH147|  arrival|               EDDN|             EDDF|               30.0|
|   EDDF| NJE413F|  arrival|               EDDL|             EDDF|               30.0|
|   EDDF|  DLH1AN|  arrival|               EDDS|             EDDF|               30.0|
|   EDDF|  DLH048|departure|               EDDF|             EDDV|               30.0|
|   EDDF|   DHXCG|departure|               EDDF|             EDGJ|               30.0|
|   EDDF|  DLH1AN|  arrival|               EDDS|             EDDF|               30.0|
|   EDDF|  DLH078|departure|               EDDF|             EDDL|               

#### 2.5.5 Top rute (perechi departure → arrival) cele mai frecvente

Identificăm cele mai frecvente rute din setul de date, adică perechile de aeroporturi între care există cel mai mult trafic.

In [26]:
from pyspark.sql.functions import coalesce

df_routes = df.withColumn(
    'plecare',
    when(col('type') == 'arrival', col('estDepartureAirport'))
    .otherwise(col('airport'))
).withColumn(
    'sosire',
    when(col('type') == 'arrival', col('airport'))
    .otherwise(col('estArrivalAirport'))
).filter(
    col('plecare').isNotNull() & col('sosire').isNotNull()
).filter(
    col('plecare') != col('sosire')
)

df_routes.groupBy('plecare', 'sosire') \
         .agg(count('*').alias('frecventa')) \
         .orderBy('frecventa', ascending=False) \
         .show(15)

+-------+------+---------+
|plecare|sosire|frecventa|
+-------+------+---------+
|   KJFK|  EGLL|     1480|
|   EGLL|  KJFK|     1462|
|   EHAM|  EGLL|     1440|
|   LFPG|  EDDF|     1399|
|   EGLL|  EHAM|     1320|
|   EGLL|  LFPG|     1232|
|   LFPG|  EGLL|     1196|
|   EDDF|  EGLL|     1189|
|   EDDF|  LFPG|     1177|
|   EGLL|  EDDF|     1151|
|   LFPG|  EHAM|     1090|
|   EHAM|  LFPG|     1034|
|   EHAM|  EDDF|      903|
|   EDDF|  EHAM|      878|
|   LOWW|  EDDF|      851|
+-------+------+---------+
only showing top 15 rows



### 2.6 Analiză cu Spark SQL

Înregistrăm DataFrame-ul ca view temporar pentru a putea folosi Spark SQL.

#### 2.6.1 Cele mai aglomerate ore per aeroport

Identificăm top 3 cele mai aglomerate ore pentru fiecare aeroport, calculând media zborurilor pe oră de-a lungul tuturor zilelor colectate.

In [27]:
df_hourly.createOrReplaceTempView('flights_hourly')
df.createOrReplaceTempView('flights')

from pyspark.sql.functions import rank
from pyspark.sql import Window

w = Window.partitionBy('airport').orderBy(col('medie_zboruri_pe_ora').desc())

spark.sql("""
    SELECT
        airport,
        arrival_hour,
        ROUND(AVG(flights_count), 1) AS medie_zboruri_pe_ora,
        MAX(flights_count)           AS maxim_zboruri
    FROM flights_hourly
    GROUP BY airport, arrival_hour
""").withColumn('rank', rank().over(w)) \
    .filter(col('rank') <= 3) \
    .orderBy('airport', 'rank') \
    .show()

+-------+------------+--------------------+-------------+----+
|airport|arrival_hour|medie_zboruri_pe_ora|maxim_zboruri|rank|
+-------+------------+--------------------+-------------+----+
|   EDDF|          18|                65.1|           79|   1|
|   EDDF|          17|                59.7|           73|   2|
|   EDDF|          20|                52.0|           66|   3|
|   EGLL|          20|                76.1|           93|   1|
|   EGLL|          18|                72.8|           88|   2|
|   EGLL|          19|                67.0|           79|   3|
|   EGLL|          17|                67.0|           78|   3|
|   EHAM|          17|                87.0|          102|   1|
|   EHAM|          20|                73.3|           89|   2|
|   EHAM|          18|                64.6|           86|   3|
|   LFPG|          20|                68.3|           86|   1|
|   LFPG|          17|                63.8|           79|   2|
|   LFPG|          18|                62.8|           7

#### 2.6.2 Comparație trafic weekend vs weekday per aeroport

Analizăm dacă există diferențe semnificative între traficul din zilele lucrătoare și weekend pentru fiecare aeroport monitorizat.

In [28]:
spark.sql("""
    SELECT
        airport,
        CASE WHEN is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END AS tip_zi,
        ROUND(AVG(flights_count), 1) AS medie_zboruri_pe_ora,
        SUM(flights_count)           AS total_zboruri
    FROM flights_hourly
    GROUP BY airport, is_weekend
    ORDER BY airport, tip_zi
""").show()

+-------+-------+--------------------+-------------+
|airport| tip_zi|medie_zboruri_pe_ora|total_zboruri|
+-------+-------+--------------------+-------------+
|   EDDF|Weekday|                28.2|        33292|
|   EDDF|Weekend|                28.0|        12532|
|   EGLL|Weekday|                34.3|        41159|
|   EGLL|Weekend|                34.3|        15491|
|   EHAM|Weekday|                30.9|        37046|
|   EHAM|Weekend|                30.5|        13801|
|   LFPG|Weekday|                33.4|        40106|
|   LFPG|Weekend|                31.4|        14209|
|   LROP|Weekday|                 8.7|         9504|
|   LROP|Weekend|                 8.6|         3568|
+-------+-------+--------------------+-------------+



#### 2.6.3 Top 10 rute internaționale cele mai frecvente

Identificăm cele mai frecvente rute internaționale din setul de date, excluzând zborurile unde aeroportul de plecare este același cu cel de sosire.

In [29]:
spark.sql("""
    SELECT
        estDepartureAirport AS plecare,
        estArrivalAirport   AS sosire,
        COUNT(*)            AS nr_zboruri
    FROM flights
    WHERE estDepartureAirport IS NOT NULL
      AND estArrivalAirport   IS NOT NULL
      AND estDepartureAirport != estArrivalAirport
    GROUP BY estDepartureAirport, estArrivalAirport
    ORDER BY nr_zboruri DESC
    LIMIT 10
""").show()

+-------+------+----------+
|plecare|sosire|nr_zboruri|
+-------+------+----------+
|   KJFK|  EGLL|      1480|
|   EGLL|  KJFK|      1462|
|   EHAM|  EGLL|      1440|
|   LFPG|  EDDF|      1399|
|   EGLL|  EHAM|      1320|
|   EGLL|  LFPG|      1232|
|   LFPG|  EGLL|      1196|
|   EDDF|  EGLL|      1189|
|   EDDF|  LFPG|      1177|
|   EGLL|  EDDF|      1151|
+-------+------+----------+



### 2.7 Salvarea tabelului agregat

Salvăm tabelul agregat `df_hourly` ca CSV pentru a fi folosit în notebooks-urile următoare (ML și DL).

In [30]:
output_path = os.path.join('..', 'data', 'hourly_traffic.csv')

df_hourly.coalesce(1) \
         .write \
         .csv(output_path, header=True, mode='overwrite')

print(f'Tabel agregat salvat in: {output_path}')
print(f'Total inregistrari: {df_hourly.count()}')

Tabel agregat salvat in: ..\data\hourly_traffic.csv
Total inregistrari: 8088


In [60]:
spark.stop()
print('SparkSession oprit!')

SparkSession oprit!
